In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import pickle

articles = pd.read_csv("../data/articles.csv")
transactions = pd.read_csv("../data/transactions_train.csv")

In [2]:
def create_article_description(row):
    parts = [
        row.get("product_type_name", ""),
        row.get("colour_group_name", ""),
        row.get("department_name", ""),
        row.get("section_name", ""),
        row.get("garment_group_name", "")
    ]
    return " ".join([str(x) for x in parts if pd.notna(x)])

articles["description"] = articles.apply(create_article_description, axis=1)

articles[["article_id", "description"]].head()


,article_id,description
0,108775015,Vest top Black Jersey Basic Womens Everyday Ba...
1,108775044,Vest top White Jersey Basic Womens Everyday Ba...
2,108775051,Vest top Off White Jersey Basic Womens Everyda...
3,110065001,Bra Black Clean Lingerie Womens Lingerie Under...
4,110065002,Bra White Clean Lingerie Womens Lingerie Under...


In [3]:
vectorizer = TfidfVectorizer(max_features=5000)
article_vectors = vectorizer.fit_transform(articles["description"])

retrieval_model = NearestNeighbors(n_neighbors=100, metric="cosine")
retrieval_model.fit(article_vectors)

print("Article vector shape:", article_vectors.shape)

Article vector shape: (105542, 303)


In [4]:
test_customer = transactions["customer_id"].value_counts().index[0]

user_history = transactions[transactions["customer_id"] == test_customer]["article_id"].tail(5)
print("Customer recent purchases:", list(user_history))


Customer recent purchases: [886566001, 886566001, 904965002, 926502001, 879291001]


In [7]:
article_id_to_index = pd.Series(articles.index, index=articles["article_id"]).to_dict()

history_indices = [
    article_id_to_index[a]
    for a in user_history
    if a in article_id_to_index
]

user_vector = np.asarray(article_vectors[history_indices].mean(axis=0))
distances, indices = retrieval_model.kneighbors(user_vector, n_neighbors=20)

candidates = articles.iloc[indices[0]][["article_id", "description"]]
candidates.head(10)

,article_id,description
102417,903850001,Dress Black Dresses Divided Collection Dresses...
91749,850985001,Dress Black Dresses Divided Collection Dresses...
58985,721744002,Dress Black Dresses Divided Collection Dresses...
58984,721744001,Dress Black Dresses Divided Collection Dresses...
91746,850984001,Dress Black Dresses Divided Collection Dresses...
91748,850984004,Dress Black Dresses Divided Collection Dresses...
60023,726227001,Dress Black Dresses Divided Collection Dresses...
61042,731996001,Dress Black Dresses Divided Collection Dresses...
102426,903868001,Dress Black Dresses Divided Collection Dresses...
102427,903868003,Dress Black Dresses Divided Collection Dresses...


In [8]:
pickle.dump(vectorizer, open("../data/tfidf_vectorizer.pkl", "wb"))
pickle.dump(retrieval_model, open("../data/retrieval_model.pkl", "wb"))
articles[["article_id", "description"]].to_csv("../data/articles_with_descriptions.csv", index=False)

## Retrieval layer explanation

The retrieval layer narrows the full catalog down to a smaller candidate set. Instead of scoring every product with a ranking model, we first find products that are similar to the user's recent purchases. This is similar to fraud systems where a fast filter or rule narrows the universe before a heavier model scores the highest-risk events.